## 😘 History of Robotics
---

classical robotics -> reinforcement robotic learning -> behaviour cloning -> scalable architechture for robotics: VLA


## 😁 LeRobot
---
### 1. Definition
Essentially, it is a Python library for robotic learning, lowering the barrier of robotics development.

### 2. Important Concepts
- #### a standardized approach for connecting to different robot platforms ?
  - first the class for collating data, `LeRobotDataset`, defines a standardized and efficient to access data;

- #### `LeRobotDataset` ?
  A special dataset class for robotic data, which can access the multimodal and temporal robotic data efficiently.

- #### separation of planning and execution ?
  

- #### Supported Robots ? 
  SO-100/SO-101 (3D‑printable arms) and ALOHA/ALOHA‑2 (bimanual manipulation)


## 😍 LeRobotDataset
---
### 数据集的三大构成数据类型
- Tabular data: 储存低维、高频数据，如关节状态、action、传感器读数等，以 parquet 形式储存。  
- Video data: 储存摄像头的图像数据，以 MP4 形式储存。  
- Meta data: 告诉程序如何解析数据的元数据。

### 数据集的基础结构
``` bash
dataset/
├─ meta/
│  ├─ info.json    # 完整数据集范式，定义所有观测、动作、图像的维度、类型、存储路径模板，另外还有fps、LeRobot 版本等。
│  ├─ stats.json   # 存储全数据集每个特征的 mean /std/min /max，训练 policy model 时标准化数据。  
│  ├─ tasks.jsonl  # 示例：每一行一条json，{"task_index": 0, "task_description": "Grab the black cube"}  
│  └─ episodes/    # 多 episodes 合并存到一个 parquet 中，分 parquet 索引  
├─ data/  
└─ videos/  
```
直接使用官方文档里的录制指令，执行完毕后数据集文件夹自动生成完整 meta 全套文件，无需手动操作；如果需要给自有机器人自定义数据集，则需要按照这个结构来手动构建 metadata。

### 在 Pytorch 框架中的调用

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

repo_id = "yaak-ai/L2D-v3"

# Load from the Hugging Face Hub (will be cached locally)
dataset = LeRobotDataset(repo_id)

# Get the 100th frame in the dataset by index
sample = dataset[100]
print(sample)
# The sample is a dictionary of tensors
# {
#     'observation.state': tensor([...]),
#     'action': tensor([...]),
#     'observation.images. front_left': tensor([C, H, W]),
#     'timestamp': tensor(1.234),
#     ...
# }
delta_timestamps = {
    "observation.images.front_left": [-0.2, -0.1, 0.0]  # 0.2 and 0.1 seconds *before* any observation
}
dataset = LeRobotDataset(
    repo_id
    delta_timestamps=delta_timestamps
)

# Accessing an index now returns a stack of frames for the specified key
sample = dataset[100]

# The image tensor will now have a time dimension
# 'observation.images.wrist_camera' has shape [T, C, H, W], where T=3
print(sample['observation.images.front_left'].shape)

batch_size=16
# Wrap the dataset in a DataLoader to process it in batches for training purposes
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size
)

# 3. Iterate over the DataLoader in a training loop
num_epochs = 1
device = "cuda" if torch.cuda.is_available() else "cpu"

for epoch in range(num_epochs):
    for batch in data_loader:
        # 'batch' is a dictionary where each value is a batch of tensors.
        # For example, batch['action'] will have a shape of [32, action_dim].

        # If using delta_timestamps, a batched image tensor might have a
        # shape of [32, T, C, H, W].

        # Move data to the appropriate device (e.g., GPU)
        observations = batch['observation.state.vehicle'].to(device)
        actions = batch['action.continuous'].to(device)
        images = batch['observation.images.front_left'].to(device)

        # Next do amazing_model.forward(batch)
        ...
